Task
1. Creat a RAG pipeline that can take following text and answer following questions
2. Try different types of chunking to get better answers?
3. Does asking questions differently give better answers? Why?
4. Try a different similarity search instead of cosine similarity - do the answers improve?



In [6]:
sample_text = """
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. It spans across nine countries.
Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization.
Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter.
Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall.
Efforts to protect the Amazon include international agreements, conservation programs, and sustainable development projects that aim to balance economic growth with environmental protection.
"""

In [9]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [11]:
import nltk
nltk.download('punkt')  # Make sure 'punkt' is downloaded

# Tokenize the sample_text into sentences
sentences = nltk.sent_tokenize(sample_text)

# Print the tokenized sentences to verify
print(sentences)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['\nThe Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers.', 'It spans across nine countries.', 'Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization.', 'Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter.', 'Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall.', 'Efforts to protect the Amazon include international agreements, conservation programs, and sustainable development projects that aim to balance economic growth with environmental protection.']


In [13]:
questions = [
    "What is the Amazon rainforest?",
    "Which countries does the Amazon span across?",
    "Why is deforestation a problem in the Amazon?",
    "How does the Amazon rainforest affect global weather patterns?",
    "What role do indigenous tribes play in the Amazon?",
    "What is the importance of the Amazon River?",
    "What types of wildlife can be found in the Amazon?",
    "How does deforestation contribute to climate change?",
    "What efforts are being made to protect the Amazon?",
    "Why is the Amazon considered a major carbon sink?"
]

In [14]:
# Combine sentences and questions
all_text = sentences + questions

# Print the combined text to verify
print(all_text)

['\nThe Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers.', 'It spans across nine countries.', 'Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization.', 'Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter.', 'Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall.', 'Efforts to protect the Amazon include international agreements, conservation programs, and sustainable development projects that aim to balance economic growth with environmental protection.', 'What is the Amazon rainforest?', 'Which countries does the Amazon span across?', 'Why is deforestation a problem in the Amazon?', 'How does the Amazon rainforest affect global weather patter

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the combined text data (sentences + questions)
X = vectorizer.fit_transform(all_text)

# Print the shape of the resulting matrix to check how many features you have
print(X.shape)

(16, 107)


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between all pairs of sentences and questions
cos_sim_matrix = cosine_similarity(X)

# Extract the similarity between the questions and the sentences
question_similarities = cos_sim_matrix[len(sentences):, :len(sentences)]

# Print similarity matrix for inspection
print(question_similarities)

[[0.47076793 0.         0.11897865 0.04588918 0.07161717 0.04483758]
 [0.07301714 0.35079998 0.03119205 0.02895472 0.13416209 0.02829119]
 [0.19549127 0.         0.1655638  0.07752146 0.09741228 0.03147166]
 [0.2130389  0.         0.02704769 0.02510763 0.27064088 0.02453226]
 [0.10975119 0.         0.02752427 0.21241349 0.15578273 0.02496452]
 [0.18390727 0.         0.15562363 0.10264437 0.07731853 0.04356634]
 [0.09893208 0.         0.07326887 0.10041322 0.06965384 0.02250355]
 [0.         0.         0.17008491 0.         0.         0.09609383]
 [0.06223474 0.         0.13043508 0.024679   0.03851539 0.26073948]
 [0.10913207 0.         0.06587083 0.02540589 0.03964983 0.02482369]]


In [18]:
# Print top 3 most similar sentences for each question based on Cosine similarity
for i, question in enumerate(questions):
    similarities = question_similarities[i]
    top_indices = similarities.argsort()[-3:][::-1]  # Get indices of top 3 similar sentences
    print(f"Top 3 sentences for question: '{question}'")
    for idx in top_indices:
        print(f"- {sentences[idx]} (Similarity: {similarities[idx]:.3f})")
    print("\n")

Top 3 sentences for question: 'What is the Amazon rainforest?'
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.471)
- Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization. (Similarity: 0.119)
- Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall. (Similarity: 0.072)


Top 3 sentences for question: 'Which countries does the Amazon span across?'
- It spans across nine countries. (Similarity: 0.351)
- Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall. (Similarity: 0.134)
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers.

In [19]:
# Example of rephrased questions
rephrased_questions = [
    "Can you describe the Amazon rainforest?",
    "In what ways does deforestation affect the climate?",
    # Add more rephrased questions as needed
]

# Combine original and rephrased questions
combined_questions = questions + rephrased_questions

# Vectorize the combined questions
X_combined = vectorizer.fit_transform(sentences + combined_questions)

# Calculate similarity
cos_sim_matrix_combined = cosine_similarity(X_combined)

# Extract and print the similarity between the questions and the sentences
combined_similarities = cos_sim_matrix_combined[len(sentences):, :len(sentences)]

# Print top 3 most similar sentences for each rephrased question
for i, question in enumerate(rephrased_questions):
    similarities = combined_similarities[i]
    top_indices = similarities.argsort()[-3:][::-1]
    print(f"Top 3 sentences for rephrased question: '{question}'")
    for idx in top_indices:
        print(f"- {sentences[idx]} (Similarity: {similarities[idx]:.3f})")
    print("\n")

Top 3 sentences for rephrased question: 'Can you describe the Amazon rainforest?'
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.453)
- Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization. (Similarity: 0.125)
- Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall. (Similarity: 0.070)


Top 3 sentences for rephrased question: 'In what ways does deforestation affect the climate?'
- It spans across nine countries. (Similarity: 0.357)
- Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall. (Similarity: 0.134)
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approxim

In [20]:
import numpy as np

# Jaccard similarity function
def jaccard_similarity(str1, str2):
    words1 = set(str1.split())
    words2 = set(str2.split())
    return len(words1.intersection(words2)) / len(words1.union(words2))

# Compute Jaccard similarity for each question-text pair
jac_sim = np.array([[jaccard_similarity(q, t) for t in sentences] for q in questions])

# Print top 3 most similar sentences for each question based on Jaccard similarity
for i, question in enumerate(questions):
    similarities = jac_sim[i]
    top_indices = similarities.argsort()[-3:][::-1]  # Get indices of top 3 similar sentences
    print(f"Top 3 sentences for question: '{question}'")
    for idx in top_indices:
        print(f"- {sentences[idx]} (Similarity: {similarities[idx]:.3f})")
    print("\n")

Top 3 sentences for question: 'What is the Amazon rainforest?'
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.176)
- Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter. (Similarity: 0.087)
- Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization. (Similarity: 0.083)


Top 3 sentences for question: 'Which countries does the Amazon span across?'
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.100)
- Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter. (Similarity: 0.080)
- Scientists believe that the Amazon plays a crucial role in global weather pat

In [21]:
# Print top 3 most similar sentences for each question based on Cosine similarity and Jaccard similarity
for i, question in enumerate(questions):
    similarities_cosine = question_similarities[i]
    similarities_jaccard = jac_sim[i]

    top_indices_cosine = similarities_cosine.argsort()[-3:][::-1]
    top_indices_jaccard = similarities_jaccard.argsort()[-3:][::-1]

    print(f"Top 3 sentences for question: '{question}'")
    print("Based on Cosine similarity:")
    for idx in top_indices_cosine:
        print(f"- {sentences[idx]} (Similarity: {similarities_cosine[idx]:.3f})")

    print("\nBased on Jaccard similarity:")
    for idx in top_indices_jaccard:
        print(f"- {sentences[idx]} (Similarity: {similarities_jaccard[idx]:.3f})")

    print("\n")

Top 3 sentences for question: 'What is the Amazon rainforest?'
Based on Cosine similarity:
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.471)
- Deforestation is a significant threat to the Amazon, with thousands of square kilometers lost each year due to agriculture, logging, and urbanization. (Similarity: 0.119)
- Scientists believe that the Amazon plays a crucial role in global weather patterns by releasing water vapor into the atmosphere, which influences rainfall. (Similarity: 0.072)

Based on Jaccard similarity:
- 
The Amazon rainforest is the largest tropical rainforest in the world, covering approximately 5.5 million square kilometers. (Similarity: 0.176)
- Indigenous tribes have lived in the Amazon for thousands of years, relying on its rich biodiversity for food, medicine, and shelter. (Similarity: 0.087)
- Deforestation is a significant threat to the Amazon, with thousands of squa

In [23]:
##Rephrasing questions can lead to better answers because it provides more context and clarification in the question, which helps the retrieval system.

##Cosine Similarity is the better measure for this task, as it works well with both the original and rephrased questions and is more suited to text retrieval and semantic matching.

##Jaccard Similarity might not work as well for rephrased questions because it relies heavily on exact word overlap, which is less likely to happen with rephrased questions.